# ⚡ NVIDIA Nemotron-3-Nano-30B Batch Inference using vLLM

This notebook runs fast batch inference on the reasoning challenge dataset using **vLLM**. It parses the model's intermediate thoughts (wrapped in `<think>` tags) and final answers, compares them to ground truth, and outputs a diagnostic log.

### 🚀 Why vLLM?
* **High Throughput**: vLLM uses PagedAttention and continuous batching, which processes the dataset 10x-30x faster than standard Hugging Face `.generate()` loops.
* **Matches Competition Evaluation**: The leaderboard evaluates your submissions using vLLM, so running offline inference here mirrors the evaluation environment precisely.
* **LoRA Support**: This script contains code to optionally load your custom trained LoRA adapters directly into the vLLM engine.

## 📦 Step 0: Offline Wheels Installation

We install all the pre-downloaded offline wheels (including vLLM and its dependencies) from the dataset directory.

In [ ]:
import os
import glob
import subprocess

# Install all local packages offline from the ml-wheels directory
# Check paths in order of specificity
paths_to_check = [
    "/kaggle/input/datasets/ckskaggle/vllm-offline-06-26/vllm-offline/",
    "/kaggle/input/vllm-offline-06-26/vllm-offline/",
    "/kaggle/input/vllm-offline/",
    "/kaggle/input/datasets/ckskaggle/ml-wheels/"
]
wheels_dir = None
for p in paths_to_check:
    if os.path.exists(p):
        wheels_dir = p
        break
if wheels_dir is None:
    wheels_dir = "/kaggle/input/datasets/ckskaggle/ml-wheels/"

if os.path.exists(wheels_dir):
    print(f"Found offline wheels directory: {wheels_dir}")
    wheel_files = glob.glob(os.path.join(wheels_dir, "*.whl"))
    if wheel_files:
        print(f"Found {len(wheel_files)} wheels. Installing offline...")
        # Install all wheels together so pip resolves local inter-dependencies correctly
        cmd = f"pip install --no-index --find-links={wheels_dir} " + " ".join(wheel_files)
        print(f"Running: {cmd}")
        try:
            subprocess.run(cmd, shell=True, check=True)
            print("All offline wheels installed successfully!")
        except Exception as e:
            print(f"Offline installation failed: {e}")
    else:
        print("No wheel (.whl) files found in the directory.")
else:
    print("Offline wheels directory not found. Assuming environment is already set up.")

## 🛠️ Step 1: Environment Configuration

In [ ]:
import os
import shutil
import stat
import glob
import site
import sys

# --- 1. CRITICAL NVIDIA / TRITON PTXAS PERMISSION PATCH ---
print("Configuring Triton binary overrides...")
candidates = glob.glob("/kaggle/usr/lib/notebooks/*/nvidia*utility*script/triton/backends/nvidia/bin/ptxas-blackwell")
if candidates:
    src = candidates[0]
    dst_dir = "/tmp/triton-bin"
    os.makedirs(dst_dir, exist_ok=True)
    dst = f"{dst_dir}/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    os.environ["TRITON_PTXAS_PATH"] = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst
    os.environ["PATH"] = f"{dst_dir}:" + os.environ["PATH"]
    os.environ["CUDA_MODULE_LOADING"] = "LAZY"
    os.environ["TRITON_DISABLE_AUTOTUNE"] = "1"
    os.environ["USE_TRITON"] = "0"
    os.environ["PTXAS_PATH"] = dst
    print(f"PTXAS override successful: {dst}")
else:
    print("ptxas-blackwell not found in utility scripts. Assuming other architecture or local run.")

# --- 2. CRITICAL CACHING DIRECTORY REDIRECTS ---
print("Configuring caching directories to /tmp...")
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TORCH_EXTENSIONS_DIR"] = "/tmp/torch_extensions"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"

# --- 3. NVIDIA CUTLASS PATH CONFIGURATION ---
search_path = "/kaggle/usr/lib/notebooks/*/nvidia*utility*script/nvidia_cutlass_dsl/python_packages/"
candidates = glob.glob(search_path)
if candidates:
    cutlass_pkg_path = candidates[0]
    site.addsitedir(cutlass_pkg_path)
    print(f"NVIDIA Cutlass DSL package path loaded from: {cutlass_pkg_path}")

## 📥 Step 2: Import Dependencies

In [ ]:
import torch
import re
import polars as pl
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
import kagglehub

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")

## 🤖 Step 3: Load Model with vLLM Engine

In [ ]:
# --- CONFIGURATION ---
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
ADAPTER_PATH = None  # If you want to test a trained adapter, set this path (e.g. "/kaggle/working")

# Set to 2 if running on Kaggle 2x T4 GPUs, or 1 for single L4/A100 GPU
TENSOR_PARALLEL_SIZE = 1 

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

print("Initializing vLLM Engine...")
llm = LLM(
    model=MODEL_PATH,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    trust_remote_code=True,
    max_model_len=8192,
    gpu_memory_utilization=0.85,
    enable_lora=(ADAPTER_PATH is not None),  # Enable LoRA if adapter path is provided
    max_loras=1,
    max_lora_rank=32
)

print("vLLM initialized successfully.")

## 🛠️ Step 4: Define Output Parsing & Answer Extraction Helpers

In [ ]:
def extract_boxed_content(text):
    """
    Extracts the content inside \boxed{...} LaTeX blocks.
    Prioritizes the last matching boxed structure, handling basic nested brackets.
    """
    matches = re.findall(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}", text)
    if matches:
        return matches[-1].strip()
    
    # Fallback heuristics
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    if lines:
        last_line = lines[-1]
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", last_line)
        if numbers:
            return numbers[-1]
        return last_line
    return ""

def parse_model_output(output_text):
    """
    Parses the generated response to separate thinking process and the final answer.
    """
    thought = ""
    answer_raw = output_text
    
    if "<think>" in output_text:
        parts = output_text.split("<think>", 1)
        if "</think>" in parts[1]:
            thought_parts = parts[1].split("</think>", 1)
            thought = thought_parts[0].strip()
            answer_raw = thought_parts[1].strip()
        else:
            thought = parts[1].strip()
            answer_raw = ""
            
    extracted_answer = extract_boxed_content(answer_raw)
    return thought, answer_raw, extracted_answer

## ⚡ Step 5: Fast Batch Inference Loop

In [ ]:
# --- DATA CONFIGURATION ---
data_path = "DATA/train.csv" if os.path.exists("DATA/train.csv") else "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pl.read_csv(data_path)

# Let's run a larger sample of 200 rows for diagnosis (since vLLM is incredibly fast!)
DIAGNOSTIC_SAMPLES = 200
df_sample = df.head(DIAGNOSTIC_SAMPLES)

print(f"Preparing prompts for {len(df_sample)} examples...")
prompts = []
for row in df_sample.iter_rows(named=True):
    prompt_text = row["prompt"]
    formatted_prompt = prompt_text + "\nReason step-by-step and write your final answer inside \\boxed{}."
    
    messages = [{"role": "user", "content": formatted_prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(input_text)

# Set sampling parameters
sampling_params = SamplingParams(
    temperature=0.0,  # Greedier decoding
    top_p=1.0,
    max_tokens=2048
)

print("Running batch generation via vLLM...")
lora_request = LoRARequest("adapter", 1, ADAPTER_PATH) if ADAPTER_PATH is not None else None

# Generate completions
outputs = llm.generate(
    prompts,
    sampling_params,
    lora_request=lora_request
)

print("Batch inference complete! Parsing results...")
results = []
for idx, output in enumerate(outputs):
    row = df_sample.row(idx, named=True)
    generated_text = output.outputs[0].text
    
    thought, raw_response, final_answer = parse_model_output(generated_text)
    
    is_correct = (final_answer.strip() == row["answer"].strip())
    
    results.append({
        "id": row.get("id", ""),
        "prompt": row["prompt"],
        "ground_truth": row["answer"],
        "model_thought": thought,
        "model_raw_response": raw_response,
        "model_answer": final_answer,
        "is_correct": is_correct
    })

# Convert results to Polars DataFrame
df_results = pl.DataFrame(results)

# Calculate and print accuracy
accuracy = df_results["is_correct"].mean()
print(f"\nDiagnostic Accuracy on {DIAGNOSTIC_SAMPLES} samples: {accuracy * 100:.2f}%")

# Save diagnostic outputs
output_csv = "diagnostic_results_vllm.csv"
df_results.write_csv(output_csv)
print(f"Diagnostic results saved to: {output_csv}")

## 🛑 Step 6: View and Diagnose Failure Cases

In [ ]:
df_failures = df_results.filter(pl.col("is_correct") == False)
print(f"Total failure cases: {len(df_failures)}\n")

# Display first 5 failures to analyze
for i in range(min(5, len(df_failures))):
    row = df_failures.row(i, named=True)
    print(f"❌ FAILURE CASE {i+1} (ID: {row['id']})")
    print(f"Prompt: {row['prompt']}")
    print(f"Ground Truth: {row['ground_truth']}")
    print(f"Model Extracted Answer: {row['model_answer']}")
    print(f"Model Thought Process:\n{row['model_thought']}")
    print("-"*80)